## Simple Mission in Auto-Mode with collition Avoidance

This notebook serves to check the ardupilot installation and debugging

In [ ]:
import os
import threading
import time

from simulator.config import (
    ARDU_LOGS_PATH,
    ARDUPILOT_VEHICLE_PATH,
    ENV_CMD_PYT,
    VEH_PARAMS_PATH,
    BasePort,
)
from simulator.helpers.cleanup import clean
from simulator.helpers.connections import create_tcp_conn, wait_for_port
from simulator.helpers.coordinates import ENU, GRAPose
from simulator.helpers.processes import create_process
from simulator.helpers.setup_log import setup_logging
from simulator.planner import AutoPlan, Plan, State
from simulator.vehicle.router import MAVLinkRouter
from simulator.vehicle.state import VehicleState

clean()

## Launch Copter (ardupilot)

In [ ]:
gra_origin = GRAPose(lat=-35.3633245,lon=149.1652241,alt=0,heading=0)
spawn_str = gra_origin.to_str()

In [ ]:
enu_pos = ENU(x=15,y=15,z=5)
intruder_pos = gra_origin.to_abs(enu_pos)

In [ ]:
sim_vehicle_path = os.path.expanduser(ARDUPILOT_VEHICLE_PATH)
sysid = 1

# set up linked virtual serial port devices for this copter
create_process(
    (
        f"socat -d -d pty,raw,echo=0,link=/tmp/adsb_{sysid}_ardupilot pty,raw,echo=0,link=/tmp/adsb_{sysid}_injector"
    ),
    after="exec bash",
    visible= False,
    suppress_output=True,
    title="socat",
)

# set up fake ADS-B device
create_process(
    (
        f"python3 adsb_injector.py "
        f"--lat {intruder_pos.lat} "
        f"--lon {intruder_pos.lon} "
        f"--alt {intruder_pos.alt} "
        f"--uart /tmp/adsb_{sysid}_injector"
    ),
    after="exec bash",
    visible= False,
    suppress_output=True,
    title="adsb_injector",
)

vehicle_cmd = (
    f"python3 {sim_vehicle_path} "
    f"-v ArduCopter "
    f'-A "--serial5=uart:/tmp/adsb_{sysid}_ardupilot:57600" '
    f"-I0 --sysid {sysid} "
    f"--no-rebuild "
    f"--use-dir={ARDU_LOGS_PATH} "
    f"--add-param-file {VEH_PARAMS_PATH} "
    f"--no-mavproxy "
    f'--custom-location={spawn_str}'
)

create_process(
                vehicle_cmd,
                after="exec bash",
                visible= False,
                suppress_output=True,
                title="ardu_vehicle",
                env_cmd=ENV_CMD_PYT,
            )
wait_for_port(port = BasePort.ARP, timeout = 0.5)



## Connect to the vehicle

In [ ]:
conn = create_tcp_conn(
    base_port=BasePort.ARP, 
    offset=0,
    role="client", 
    src_sysid=sysid,
    src_compid=140
)

print("✅ TCP connection established!")

## Start the router

In [ ]:
vehicle_state = VehicleState.create()
router_stop = threading.Event()
router = MAVLinkRouter(
    conn=conn,
    state=vehicle_state,
    stop_event=router_stop
)
router.start()

print("✅ MAVLink router started")

## Create auto plan

In [ ]:
mission_path = "simulator/planner/missions/square_mission.waypoints"
plan = AutoPlan(name="basic_auto_plan", mission_path=mission_path)
plan

# Save mission

In [ ]:
enu_path = Plan.create_square_path(side_len=10, alt=5, clockwise=True)
gra_path=GRAPose.unpose_all(gra_origin.to_abs_all(enu_path))
plan.save_basic_mission(sysid=sysid,gra_wps=gra_path)

# Execute Auto Plan

In [ ]:
setup_logging('plan',verbose=1)
plan.bind(conn,gra_origin.unpose(), vehicle_state)
while plan.state != State.DONE:
    plan.act()
    time.sleep(0.1)
    
router_stop.set()
router.join(timeout=1)
conn.close()

In [ ]:
plan